# EMRI chain post-processing

Loads the Eryn HDF backend from `emri_test_script_td_wave.py` and walks through the usual sanity checks: trace plots, acceptance fraction, autocorrelation, and a corner plot with injected values overlaid.

Chain dims for this run: `ntemps=2`, `nwalkers=2`, 12 sampled parameters in the order:
`M, mu, a, p0, e0, dist, cosqS, phiS, cosqK, phiK, Phi_phi0, Phi_r0`.

Two parameters are held fixed in the sampler via `fill_dict` and so are not in the chain: `x0` (prograde flag) and `Phi_theta0` (ignored in equatorial).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from eryn.backends import HDFBackend

CHAIN_PATH = os.path.expanduser(
    "~/Documents/LISA/LISA_Sprint_2026/lisasprint26/lisa_sprint_2026/test_emri_pe.h5"
)
backend = HDFBackend(CHAIN_PATH)
print(f"Loaded: {CHAIN_PATH}")

## 1. Chain shape and bookkeeping

In [ ]:
chain = backend.get_chain()
log_like = backend.get_log_like()
log_prior = backend.get_log_prior()

emri = chain["emri"]
print("chain['emri'] shape:", emri.shape, "-> (nsteps_saved, ntemps, nwalkers, nleaves, ndim)")
print("log_like shape:    ", log_like.shape)
print("log_prior shape:   ", log_prior.shape)
print("iteration:         ", backend.iteration)

nsteps_saved, ntemps, nwalkers, nleaves, ndim = emri.shape
print(f"\nUseful: {nsteps_saved} saved steps x {nwalkers} walkers = {nsteps_saved * nwalkers} cold-chain samples per parameter.")

In [ ]:
PARAM_NAMES = [
    "M", "mu", "a", "p0", "e0", "dist",
    "cosqS", "phiS", "cosqK", "phiK",
    "Phi_phi0", "Phi_r0",
]
assert len(PARAM_NAMES) == ndim, f"name list ({len(PARAM_NAMES)}) doesn't match ndim ({ndim})"

# Color scheme used throughout this notebook
INJ_COLOR = "purple"
REC_COLOR = "orange"

# Injected values — mirror the script
_m1       = 1.0e6
_m2       = 1.0e1
_a        = 0.99
_p0       = 6.1
_e0       = 0.3
_dist     = 2.0
_lam      = 0.2538922432234
_beta     = -0.418762312
_qS       = np.pi / 2.0 - _beta
_phiS     = _lam
_qK       = 0.2340980298542
_phiK     = 4.098234232
_Phi_phi0 = 1.0803123123
_Phi_r0   = 4.32094823423

INJECTED = {
    "M":        _m1,
    "mu":       _m2,
    "a":        _a,
    "p0":       _p0,
    "e0":       _e0,
    "dist":     _dist,
    "cosqS":    np.cos(_qS),
    "phiS":     _phiS,
    "cosqK":    np.cos(_qK),
    "phiK":     _phiK,
    "Phi_phi0": _Phi_phi0,
    "Phi_r0":   _Phi_r0,
}
truth = np.array([INJECTED[n] for n in PARAM_NAMES])

## 2. Acceptance fraction

Crude diagnostic: fraction of proposed moves that were accepted, per walker, on the cold chain. For StretchMove, you want this to be roughly 0.2-0.5. Values near 0 or 1 mean the proposal is poorly scaled.

In [ ]:
try:
    af = backend.get_autocorr_time()
    print("autocorr times (per param, cold temp):")
    print(af)
except Exception as e:
    print("autocorr not yet usable:", e)

try:
    accept = backend.accepted  # shape (ntemps, nwalkers) cumulative
    print("\ncumulative acceptances (ntemps, nwalkers):\n", accept)
    print("fraction (vs iteration):", accept / max(backend.iteration, 1))
except AttributeError:
    print("backend.accepted not available in this Eryn version")

## 3. Trace plots

Cold-temperature chain (index 0). Each color is a walker; horizontal dashed line is the injected value. With only a handful of steps this just shows the chain hasn't blown up.

In [ ]:
from matplotlib.lines import Line2D

cold = emri[:, 0, :, 0, :]  # (nsteps_saved, nwalkers, ndim)

WALKER_COLORS = ["gold", "purple"]
TRACE_INJ_COLOR = "black"

fig, axes = plt.subplots(ndim, 1, figsize=(10, 1.6 * ndim), sharex=True)
for i, name in enumerate(PARAM_NAMES):
    ax = axes[i]
    for w in range(nwalkers):
        ax.plot(cold[:, w, i], lw=0.8, alpha=0.8,
                color=WALKER_COLORS[w % len(WALKER_COLORS)])
    ax.axhline(truth[i], ls="--", color=TRACE_INJ_COLOR, lw=1.0, alpha=0.9)
    ax.set_ylabel(name)
axes[-1].set_xlabel("saved step")

legend_handles = [
    Line2D([0], [0], color=WALKER_COLORS[w % len(WALKER_COLORS)], lw=1.5,
           label=f"walker {w}")
    for w in range(nwalkers)
] + [
    Line2D([0], [0], color=TRACE_INJ_COLOR, lw=1.5, ls="--", label="injected"),
]
fig.legend(handles=legend_handles, loc="upper right", bbox_to_anchor=(1.0, 1.0))
fig.suptitle("Trace plots (cold chain)", y=1.0)
fig.tight_layout()

In [ ]:
cold_ll = log_like[:, 0, :]  # (nsteps, nwalkers)
fig, ax = plt.subplots(figsize=(8, 3))
for w in range(nwalkers):
    ax.plot(cold_ll[:, w], lw=0.8, alpha=0.8,
            color=WALKER_COLORS[w % len(WALKER_COLORS)],
            label=f"walker {w}")
ax.set_xlabel("saved step")
ax.set_ylabel("log-likelihood (cold)")
ax.legend(loc="best")
ax.set_title("Cold-chain log-likelihood trace")
fig.tight_layout()

## 4. Corner plot

Flattens cold chain across walkers. With this many samples it's not a posterior — it's a scatter plot. Re-run after a longer chain for anything to interpret here.

Optional discard: drop the first `discard` steps as burn-in.

In [ ]:
from matplotlib.lines import Line2D

discard = 0
flat = cold[discard:].reshape(-1, ndim)
print("flat samples shape:", flat.shape)

try:
    import corner
    fig = corner.corner(
        flat,
        labels=PARAM_NAMES,
        truths=truth,
        show_titles=True,
        title_fmt=".3g",
        plot_density=False,
        plot_contours=False,    # set True once you have enough samples
        plot_datapoints=True,
        color=REC_COLOR,
        truth_color=INJ_COLOR,
        hist_kwargs=dict(color=REC_COLOR),
    )
    legend_handles = [
        Line2D([0], [0], color=REC_COLOR, lw=2, label="recovered"),
        Line2D([0], [0], color=INJ_COLOR, lw=2, label="injected"),
    ]
    fig.legend(handles=legend_handles, loc="upper right", bbox_to_anchor=(0.98, 0.98), fontsize=12)
    fig.suptitle(f"Corner plot — sampled basis ({flat.shape[0]} samples)", y=1.02)
except ImportError:
    print("`corner` not installed in this env — `pip install corner`")

## 5. Sampled basis -> physical basis

The sampler walks in `(cosqS, cosqK, ...)`. Undo the transforms to view the chain in the physical sky/spin polar angles.

In [ ]:
from matplotlib.lines import Line2D

qS_s = np.arccos(flat[:, PARAM_NAMES.index("cosqS")])
qK_s = np.arccos(flat[:, PARAM_NAMES.index("cosqK")])

physical = np.column_stack([
    flat[:, PARAM_NAMES.index("M")],
    flat[:, PARAM_NAMES.index("mu")],
    flat[:, PARAM_NAMES.index("a")],
    flat[:, PARAM_NAMES.index("p0")],
    flat[:, PARAM_NAMES.index("e0")],
    flat[:, PARAM_NAMES.index("dist")],
    qS_s,
    flat[:, PARAM_NAMES.index("phiS")],
    qK_s,
    flat[:, PARAM_NAMES.index("phiK")],
    flat[:, PARAM_NAMES.index("Phi_phi0")],
    flat[:, PARAM_NAMES.index("Phi_r0")],
])
PHYSICAL_NAMES = ["M", "mu", "a", "p0", "e0", "dist",
                  "qS", "phiS", "qK", "phiK",
                  "Phi_phi0", "Phi_r0"]
physical_truth = np.array([
    _m1, _m2, _a, _p0, _e0, _dist,
    _qS, _phiS, _qK, _phiK,
    _Phi_phi0, _Phi_r0,
])

try:
    import corner
    fig = corner.corner(
        physical,
        labels=PHYSICAL_NAMES,
        truths=physical_truth,
        show_titles=True,
        title_fmt=".3g",
        plot_density=False,
        plot_contours=False,
        plot_datapoints=True,
        color=REC_COLOR,
        truth_color=INJ_COLOR,
        hist_kwargs=dict(color=REC_COLOR),
    )
    legend_handles = [
        Line2D([0], [0], color=REC_COLOR, lw=2, label="recovered"),
        Line2D([0], [0], color=INJ_COLOR, lw=2, label="injected"),
    ]
    fig.legend(handles=legend_handles, loc="upper right", bbox_to_anchor=(0.98, 0.98), fontsize=12)
    fig.suptitle("Corner plot — physical basis", y=1.02)
except ImportError:
    pass

## 6. Suggestions for the next run

- **Longer chains:** `nsteps=2000` with `thin_by=1` gives 2000 saved samples per walker, but only `nwalkers=2` makes that ~4000 cold-chain samples per parameter. The script writes incrementally to `test_emri_pe.h5` so you can Ctrl-C and resume — the `if os.path.exists(fp)` branch already picks up the last sample.
- **More walkers/temps:** `nwalkers=2` is far below the StretchMove rule of thumb (>= 2*ndim, so >=24). Bump it once you trust the pipeline. EMRI likelihood evaluation is expensive, so weigh this against runtime.
- **Autocorrelation:** once you have >50x autocorr-time worth of samples, `backend.get_autocorr_time()` will stop raising and you can thin properly.
- **Sensible burn-in:** look at the log-likelihood trace and discard everything before it plateaus — set `discard` in the corner-plot cell.
- **Priors are tight (`factor=1e-3`):** the prior box around every intrinsic parameter is only +/-0.1%. Useful for testing that the sampler tracks an injection, not for real PE — broaden the priors before drawing physical conclusions.